# BrickOrSwish: train our own detector

Fine-tunes **RF-DETR Nano** (Apache-2.0, COCO-pretrained) to find 4 classes: `ball`, `ball_in_net`, `rim`, `person`.
Everything runs on Google's machine; your computer only gets the finished `model.onnx`.

| Step | Where | Time |
|---|---|---|
| 1. Frames + draft labels from your earlier runs | this notebook (no GPU needed) | minutes |
| 2. Correct the labels | labelling website (CVAT) | hours, by hand |
| 3. Build the dataset | this notebook | seconds |
| 4. Train | this notebook, **T4 GPU** | about 1-3 h (estimate) |
| 5. Export to ONNX, check it | this notebook | minutes |

**Before step 1**, put on your Google Drive (folder `brickorswish/`): the videos, and each video's run folder from
`ml/results/` with only `hoop.json`, `detections.json`, `events.json`, `camera_motion.json`, `rim.json` (a few MB;
leave out `annotated.*`, they are hundreds of MB). Then fill in `SESSIONS` below.

**Privacy:** the frames are your footage. They stay on your Drive and go to the labelling site you choose.

**Security:** same as the detection notebook: pinned versions, PyTorch >= 2.10, every dependency checked
against OSV.dev; it stops if anything is vulnerable.

**Runtime → Change runtime type → T4 GPU** before step 4. Steps 1-3 also work without a GPU.
Every step saves to your Drive, so after a disconnect: run Settings + Setup again, then continue at your step.

In [ ]:
# Settings
BRANCH = "feat/shot-logic-reference"  # branch of github.com/Pietrzaaq/swish-or-brick to run
RFDETR_VERSION = "1.12.1"             # used only until ml/requirements-colab.txt exists
ALLOW_VULNS = False                   # True = continue even if the security audit finds known vulnerabilities
DRIVE = "/content/drive/MyDrive/brickorswish"

# Videos to learn from, each with its run folder (copied from ml/results/ on your computer). Fix the paths.
SESSIONS = [
    {"name": "9min", "video": f"{DRIVE}/9min.MOV", "results": f"{DRIVE}/results/9min-local"},
    {"name": "2min", "video": f"{DRIVE}/2min.MP4", "results": f"{DRIVE}/results/2min-local"},
]
DATASET = "v1"         # this training round: everything goes to brickorswish/training/<DATASET> on Drive
DRAFTS_AGAIN = False   # True = make the step 1 frames again (they are reused otherwise)
LABELS = f"{DRIVE}/training/{DATASET}/labels.zip"   # the labelling tool's export (step 2)
EPOCHS = 50            # stops earlier when validation stops improving
BATCH, GRAD_ACCUM = 4, 4                            # T4; "out of memory" -> BATCH = 2, GRAD_ACCUM = 8
MODEL_NAME = f"brickorswish-{DATASET}"

In [ ]:
# Setup: code, Drive, detector library (1-3 minutes)
import os, sys, subprocess
from pathlib import Path
!nvidia-smi -L || echo "no GPU: fine for steps 1-3; step 4 needs Runtime > Change runtime type > T4 GPU"
!rm -rf /content/swish-or-brick && git clone -q --depth 1 -b {BRANCH} https://github.com/Pietrzaaq/swish-or-brick.git /content/swish-or-brick
!git -C /content/swish-or-brick log -1 --format="running code from commit %h: %s (%cr)"
sys.path.insert(0, "/content/swish-or-brick/ml")
if not os.path.exists("/content/swish-or-brick/ml/brickorswish/train/fit.py"):
    raise RuntimeError(f"{BRANCH} on GitHub has no training code yet: push your commits, then run this again")
PINS = "/content/swish-or-brick/ml/requirements-colab.txt"
if os.path.exists(PINS):
    !pip install -q -r {PINS}
else:
    !pip install -q rfdetr=={RFDETR_VERSION}

from google.colab import drive
drive.mount("/content/drive")  # asks you to allow access the first time
from brickorswish.train import CLASSES
ROUND = Path(DRIVE) / "training" / DATASET
DRAFTS = Path("/content/drafts")
DATASET_DIR = Path("/content/dataset")
RUN = ROUND / "run"

def run_py(*args):
    # python -m <args> in a fresh process (Colab preloads old libraries into this one), streaming its output
    proc = subprocess.Popen([sys.executable, "-u", "-m", *map(str, args)], cwd="/content/swish-or-brick/ml",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

In [ ]:
# Security audit (same as the detection notebook)
from importlib.metadata import version
from brickorswish.tools.deps_audit import version_tuple

if version_tuple(version("torch")) < (2, 10):
    print(f"torch {version('torch')} is older than 2.10, upgrading")
    !pip install -q -U "torch>=2.10" torchvision
assert version_tuple(version("torch")) >= (2, 10), f"torch {version('torch')} is still older than 2.10"
print(f"torch {version('torch')} OK")
ROOTS = ["rfdetr", "opencv-python", "opencv-python-headless", "opencv-contrib-python"]
rc = run_py("brickorswish.tools.deps_audit", *ROOTS, "--pins", "/content/requirements-colab.txt", "--fix")
if rc != 0 and not ALLOW_VULNS:
    raise RuntimeError("Audit failed (see above). Stop here, or set ALLOW_VULNS = True if you accept the risk.")
print("audit clean")

## Step 1: frames with draft labels

From each session: a frame every 0.1 s around every shot, one every 3 s in between, and a full frame every 15 s.
The drafts come from your earlier runs: ball and person boxes from the COCO detector, the rim from `hoop.json`,
and `ball_in_net` for a ball inside the net during shots the logic called a make. The rim's bracket (the fake
"ball") is left out on purpose. Results on your Drive in `training/<DATASET>/`.

In [ ]:
# Step 1: frames + draft labels (no GPU needed; about a minute per minute of video)
import shutil, zipfile
if (ROUND / "drafts.zip").exists() and not DRAFTS_AGAIN:
    print(f"drafts already made: {ROUND / 'drafts.zip'} (DRAFTS_AGAIN = True to make them again)")
else:
    shutil.rmtree(DRAFTS, ignore_errors=True)
    for s in SESSIONS:
        for key in ("video", "results"):
            if not Path(s[key]).exists():
                raise FileNotFoundError(f"{s[key]} not found: fix SESSIONS in Settings (Files panel on the left)")
        video = Path("/content") / Path(s["video"]).name
        if not video.exists():
            shutil.copy(s["video"], video)  # read once, faster than from Drive
        if run_py("brickorswish.train.drafts", video, s["results"], "--out", DRAFTS, "--session", s["name"]) != 0:
            raise RuntimeError(f"step 1 failed for {s['name']} (see above)")
    ROUND.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile("/content/images.zip", "w", zipfile.ZIP_STORED) as z:  # flat: names match the json
        for p in sorted((DRAFTS / "images").glob("*.jpg")):
            z.write(p, p.name)
    shutil.make_archive("/content/drafts", "zip", DRAFTS)
    for src in ("/content/images.zip", DRAFTS / "annotations.coco.json", DRAFTS / "check_these.txt",
                "/content/drafts.zip"):
        shutil.copy(src, ROUND / Path(src).name)
    print(f"\nOn your Drive, {ROUND}:\n  images.zip + annotations.coco.json  -> upload both to the labelling tool"
          "\n  check_these.txt  -> make frames with no ball box: look for the ball in the net"
          "\n  drafts.zip  -> used by step 3, leave it there")

In [ ]:
# Look at a few drafts (green ball, yellow ball_in_net, orange rim, blue person)
import json, random, shutil, cv2
from IPython.display import Image, display
if not DRAFTS.exists():
    shutil.unpack_archive(ROUND / "drafts.zip", DRAFTS)
coco = json.loads((DRAFTS / "annotations.coco.json").read_text())
colours = {"ball": (60, 200, 60), "ball_in_net": (0, 220, 255), "rim": (0, 140, 255), "person": (255, 120, 0)}
names = {c["id"]: c["name"] for c in coco["categories"]}
with_net = {a["image_id"] for a in coco["annotations"] if names[a["category_id"]] == "ball_in_net"}
pick = random.sample([im for im in coco["images"] if im["id"] in with_net], min(3, len(with_net)))
pick += random.sample(coco["images"], 3)
for im in pick:
    img = cv2.imread(str(DRAFTS / "images" / im["file_name"]))
    for a in coco["annotations"]:
        if a["image_id"] == im["id"]:
            x, y, w, h = map(int, a["bbox"])
            cv2.rectangle(img, (x, y), (x + w, y + h), colours[names[a["category_id"]]], 2)
    scale = 500 / max(img.shape[:2])
    small = cv2.resize(img, None, fx=scale, fy=scale)
    print(im["file_name"])
    display(Image(data=cv2.imencode(".jpg", small)[1].tobytes()))

## Step 2: correct the labels (by hand, on the CVAT website)

1. Download `images.zip` and `annotations.coco.json` from Drive (`brickorswish/training/<DATASET>/`).
2. On [app.cvat.ai](https://app.cvat.ai): **Projects → +**, add 4 labels, type **Rectangle**, names exactly:
   `ball`, `ball_in_net`, `rim`, `person`.
3. **Tasks → +**, pick the project, add `images.zip` as files, submit.
4. Open the task → **Actions → Upload annotations → COCO 1.0** → `annotations.coco.json`: the drafts appear.
5. Go through every frame (`F` next, `D` back; `N` draws a new box; select a box and `Del` removes it).
   Rules:
   - **ball**: the basketball wherever you can see it, tight box around the whole ball, even if partly hidden.
   - **ball_in_net**: the ball below the rim, *inside* the net (seen through the mesh). Box the ball, not the net.
     A ball falling **in front of** or beside the net is `ball`: that difference is what the model must learn.
   - **rim**: one tight box around the whole rim ring.
   - **person**: every person, a box around what is visible.
   - Remove wrong drafts: the rim's bracket, shoes, heads labelled as a ball. Fix boxes that are off.
   - Frames listed in `check_these.txt`: makes where the detector lost the ball. Draw `ball_in_net` if you can see it.
6. **Actions → Export task dataset → COCO 1.0** (images not needed). Put the zip on Drive as
   `brickorswish/training/<DATASET>/labels.zip` (the `LABELS` path in Settings).

Check your CVAT plan's limits before uploading; Roboflow works too (export "COCO"), but check whether its free
plan makes projects public before uploading your footage there.

In [ ]:
# Step 3: labelled frames -> dataset (train / valid / test, split by 20 s blocks)
if not DRAFTS.exists():
    shutil.unpack_archive(ROUND / "drafts.zip", DRAFTS)
if not Path(LABELS).exists():
    raise FileNotFoundError(f"{LABELS} not found: export the labels from CVAT (step 2) and put the zip there")
if run_py("brickorswish.train.dataset", LABELS, "--drafts", DRAFTS, "--out", DATASET_DIR) != 0:
    raise RuntimeError("step 3 failed (see above)")

In [ ]:
# Step 4: train (T4 GPU). Checkpoints go to your Drive; after a disconnect run Settings, Setup, step 3 and this
# cell again: it continues from the last checkpoint.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Runtime > Change runtime type > T4 GPU, then run Settings, Setup, step 3 again")
if not (DATASET_DIR / "train" / "_annotations.coco.json").exists():
    raise RuntimeError("No dataset on this machine: run step 3 first (seconds)")
print(f"GPU: {torch.cuda.get_device_name(0)}; checkpoints in {RUN}")
if run_py("brickorswish.train.fit", DATASET_DIR, "--out", RUN, "--epochs", EPOCHS,
          "--batch", BATCH, "--grad-accum", GRAD_ACCUM) != 0:
    raise RuntimeError("training failed (see above); run this cell again to continue from the last checkpoint")

In [ ]:
# Step 5: export to ONNX for your computer, and check it
# 1. model.onnx + model.json (card: input size, normalisation, our 4 classes, SHA-256)
# 2. the trained model and the ONNX file must give the same boxes on 60 crops of your first session
# 3. the ONNX file must run with ONLY onnxruntime + numpy + opencv-python-headless (your laptop's setup)
import json, shutil
from datetime import date
from google.colab import files
BEST = RUN / "checkpoint_best_total.pth"
if not BEST.exists():
    raise FileNotFoundError(f"{BEST} not found: train first (step 4)")
EXPORT = Path("/content/export")
DEST = Path(DRIVE) / "models" / MODEL_NAME
!pip install -q "onnx>=1.16,<2" onnxruntime==1.31.1
if run_py("brickorswish.tools.deps_audit", "onnx", "onnxruntime") != 0 and not ALLOW_VULNS:
    raise RuntimeError("onnx / onnxruntime have known vulnerabilities (see above); not exporting")

s = SESSIONS[0]
video = Path("/content") / Path(s["video"]).name
if not video.exists():
    shutil.copy(s["video"], video)
rim = Path(s["results"]) / "rim.json"
frame = json.loads(rim.read_text())["hoop_frame"] if rim.exists() else 0
common = ["--video", video, "--hoop", Path(s["results"]) / "hoop.json", "--hoop-frame", frame]
if run_py("brickorswish.detect.export_onnx", "export", "--out", EXPORT, "--weights", BEST,
          "--classes", ",".join(CLASSES), "--name", MODEL_NAME, *common) != 0:
    raise RuntimeError("Export or the trained-model vs ONNX comparison failed (see above). Don't use this model.")

!python -m venv --without-pip /content/ortenv
!pip --python /content/ortenv/bin/python install -q --no-deps onnxruntime==1.31.1 numpy==2.5.4 opencv-python-headless==4.14.0.94
proc = subprocess.Popen(["/content/ortenv/bin/python", "-u", "-m", "brickorswish.detect.export_onnx", "check-minimal",
                         str(EXPORT / "model.onnx"), *map(str, common)], cwd="/content/swish-or-brick/ml",
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError("The model needs more than onnxruntime + numpy + opencv (see above)")

DEST.mkdir(parents=True, exist_ok=True)
for name in ("model.onnx", "model.json", "parity.json"):
    shutil.copy(EXPORT / name, DEST / name)
card = json.loads((EXPORT / "model.json").read_text())
print(f"\nOn your Drive: {DEST}\nAdd this line to ml/models.sha256 on your computer:\n"
      f"{card['sha256']}  {MODEL_NAME}, trained in Colab {date.today().isoformat()}")

## Use it on your computer

1. From Drive `brickorswish/models/<MODEL_NAME>/`, put `model.onnx` + `model.json` into
   `ml/models/<MODEL_NAME>/` (git-ignored).
2. Add the line step 5 printed to `ml/models.sha256` and commit it: models not listed there are refused.
3. Run a video as before, with the new model:
   `.venv\Scripts\python -m brickorswish.detect.run_video videos\2min.MP4 --out results\2min-v1 --onnx models\brickorswish-v1\model.onnx`
4. Compare with the COCO model on the same video (`score.py` with your verdict). The shot logic already treats
   `ball_in_net` as the ball; using it directly for makes, and the `rim` box for the hoop, comes next.

Use videos the model was **not** trained on to judge it (a new session), or the test split's 20 s blocks.